# 02 — Train YOLOv8

Unzips the merged dataset from Drive and fine-tunes YOLOv8s.

| | |
|---|---|
| **Input** | Drive: merged_dataset.zip (from 01d) |
| **Classes** | `0: person` |
| **Model** | YOLOv8s (small — good balance of speed and accuracy) |
| **Weights saved to** | `/content/drive/MyDrive/AI_TRAINING/GreenVision/runs/` |

In [ ]:
!pip install ultralytics pyyaml -q

import os, shutil, glob, yaml
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

print('Ultralytics version:', ultralytics.__version__)

# Auto-detect GPU
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: No GPU detected. Training will be slow.')
    print('Go to Runtime > Change runtime type > T4 GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
DRIVE_ZIP = os.path.join(DRIVE_ROOT, 'merged_dataset.zip')
DRIVE_RUNS = os.path.join(DRIVE_ROOT, 'runs')
DATASET_DIR = '/content/dataset/merged'
DATA_YAML = '/content/data.yaml'

## 1. Unzip Merged Dataset

In [ ]:
if not os.path.exists(DRIVE_ZIP):
    raise SystemExit(f'merged_dataset.zip not found at {DRIVE_ZIP}. Run 01d_merge_datasets.ipynb first.')

# Clean old dataset
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

print('Unzipping merged dataset...')
!unzip -q -o {DRIVE_ZIP} -d /content/dataset

# Verify
train_imgs = len(glob.glob(f'{DATASET_DIR}/images/train/*.*'))
val_imgs = len(glob.glob(f'{DATASET_DIR}/images/val/*.*'))
train_lbls = len(glob.glob(f'{DATASET_DIR}/labels/train/*.txt'))
val_lbls = len(glob.glob(f'{DATASET_DIR}/labels/val/*.txt'))

print(f'Train: {train_imgs} images, {train_lbls} labels')
print(f'Val:   {val_imgs} images, {val_lbls} labels')

if train_lbls == 0 and val_lbls == 0:
    raise SystemExit('No labels found! Re-run 01d_merge_datasets.ipynb.')

# Write local data.yaml
config = {
    'path'  : DATASET_DIR,
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 1,
    'names' : {0: 'person'},
}
with open(DATA_YAML, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

SETTINGS.update({'datasets_dir': '/content/dataset'})
print('\nReady for training!')

## 2. Train

In [ ]:
model = YOLO('yolov8s.pt')  # pretrained on COCO, fine-tune on person-only

results = model.train(
    data        = DATA_YAML,
    epochs      = 50,
    imgsz       = 640,
    batch       = 16 if DEVICE != 'cpu' else 8,
    name        = 'person_detect_v1',
    project     = DRIVE_RUNS,
    patience    = 10,
    device      = DEVICE,
    workers     = 4 if DEVICE != 'cpu' else 2,
    cache       = True,
    save_period = 5,
)

# Auto-detect actual run directory (Ultralytics appends number if name exists)
RUN_DIR = str(results.save_dir)
print(f'\nRun saved to: {RUN_DIR}')

## 3. Training Results

In [ ]:
from IPython.display import Image as IPImage, display
import os

results_img = os.path.join(RUN_DIR, 'results.png')
if os.path.exists(results_img):
    display(IPImage(filename=results_img, width=900))
else:
    print(f'results.png not found at {results_img}')

## 4. Validate

In [ ]:
best_weights = os.path.join(RUN_DIR, 'weights', 'best.pt')

if not os.path.exists(best_weights):
    raise SystemExit(f'best.pt not found at {best_weights}')

best = YOLO(best_weights)
metrics = best.val(data=DATA_YAML, device=DEVICE)

print('\n=== Validation Metrics ===')
print(f'  mAP@0.5      : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'  Precision    : {metrics.box.mp:.4f}')
print(f'  Recall       : {metrics.box.mr:.4f}')

## 5. Export Model

In [ ]:
model_dir = os.path.join(DRIVE_ROOT, 'models')
os.makedirs(model_dir, exist_ok=True)
dst = os.path.join(model_dir, 'best.pt')

shutil.copy2(best_weights, dst)
size_mb = os.path.getsize(dst) / (1024*1024)

print(f'Best model saved to: {dst}')
print(f'Size: {size_mb:.1f} MB')
print()
print('To use in the backend, download this file and set in .env:')
print(f'  YOLO_MODEL=path/to/best.pt')
print()
print('Or use yolov8n.pt (pretrained) if not fine-tuning.')

---
## Done!

Best weights saved to:
```
/content/drive/MyDrive/AI_TRAINING/GreenVision/runs/person_detect_v1/weights/best.pt
```

Next: **03 — Inference Demo**